# Model Training
- Jigsaw Dataset(with clear different toxicity labels) trained for the baseline classifier.
- Model applied to Youtube comments and comment score determine
- A separate model to handle HINGLISH comments.
- Determine the routing of the comments based on language flag.

In [1]:
# Importing libraries
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import (
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    hamming_loss
)

In [2]:
# Importing JigSaw training dataset
df = pd.read_csv("../data/raw/jigsaw_train.csv")

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 159571 entries, 0 to 159570
Data columns (total 8 columns):
 #   Column         Non-Null Count   Dtype
---  ------         --------------   -----
 0   id             159571 non-null  str  
 1   comment_text   159571 non-null  str  
 2   toxic          159571 non-null  int64
 3   severe_toxic   159571 non-null  int64
 4   obscene        159571 non-null  int64
 5   threat         159571 non-null  int64
 6   insult         159571 non-null  int64
 7   identity_hate  159571 non-null  int64
dtypes: int64(6), str(2)
memory usage: 72.2 MB


In [4]:
## Label columns
label_cols =df.columns[2::]

In [5]:
# Dataset Metadata
print(f"Dataset Shape: {df.shape}")
print("\nClass Distribution:")
for col in label_cols:
    pos_count = df[col].sum()
    pct = (pos_count/len(df)) * 100
    print(f" {col:<15}: {pos_count:>6} positive samples ({pct:.2f}%)")

Dataset Shape: (159571, 8)

Class Distribution:
 toxic          :  15294 positive samples (9.58%)
 severe_toxic   :   1595 positive samples (1.00%)
 obscene        :   8449 positive samples (5.29%)
 threat         :    478 positive samples (0.30%)
 insult         :   7877 positive samples (4.94%)
 identity_hate  :   1405 positive samples (0.88%)


## Model Training 

In [6]:
# train and valuation split
## converting to standard ndarrays to prevent vectorized form crashing later in case fo nan
X = df["comment_text"].fillna("").to_numpy(dtype=str)
y = df[label_cols].to_numpy()
X_train, X_val, y_train, y_val = train_test_split(X,y, test_size=0.2, random_state=42)

In [7]:
## vectorizing text data adn fitting on training data and transforming validation
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=40000,
    sublinear_tf=True,
    strip_accents="unicode"
)

print("Fitting TF-IDF Vectorizer...")
X_train_vec = vectorizer.fit_transform(X_train)
X_val_vec = vectorizer.transform(X_val)

print(f"Vocabulary Size: {len(vectorizer.vocabulary_)}")

Fitting TF-IDF Vectorizer...
Vocabulary Size: 40000


In [8]:
# Model Instances
models = {
    "Logistic Regression": OneVsRestClassifier(
        LogisticRegression(class_weight="balanced", max_iter=1000, C=2.0)
    ),
    "Linear SVC": OneVsRestClassifier(
        LinearSVC(class_weight="balanced", max_iter=2000, C=0.5, random_state=42)
    )
}

results = []

In [9]:
# Model training and Performance metrics
for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train_vec, y_train)
    y_pred = model.predict(X_val_vec)
    
    # Calculate macro and micro metrics
    macro_f1 = f1_score(y_val, y_pred, average="macro")
    micro_f1 = f1_score(y_val, y_pred, average="micro")
    h_loss = hamming_loss(y_val, y_pred)
    
    # Per-label F1 scores
    per_label_f1 = f1_score(y_val, y_pred, average=None)
    
    metrics = {
        "Model": name,
        "Macro F1": round(macro_f1, 4),
        "Micro F1": round(micro_f1, 4),
        "Hamming Loss": round(h_loss, 4),
    }
    for col, f1 in zip(label_cols, per_label_f1):
        metrics[f"F1_{col}"] = round(f1, 4)
        
    results.append(metrics)

results_df = pd.DataFrame(results)
display(results_df)

Training Logistic Regression...
Training Linear SVC...


,Model,Macro F1,Micro F1,Hamming Loss,F1_toxic,F1_severe_toxic,F1_obscene,F1_threat,F1_insult,F1_identity_hate
0,Logistic Regression,0.5773,0.6972,0.0273,0.7422,0.4383,0.7862,0.3833,0.7073,0.4065
1,Linear SVC,0.5984,0.7139,0.0240,0.7498,0.4591,0.7879,0.4419,0.7088,0.4430


### Observation
- Linear SVC outperforms Logistic Regression in every metrics. Macro/Micro F1, Hamming Loss, F1 across labels.
- As the Imbalance detected earlier in the dataset and using `class_weight=balanced` to handle that. Linear SVC handles class imbalance somewhat better than Logistic Regression.
- As Linear SVC does not 
- Despite Linear SVC's better performance, Logistic Regression is
  selected going forward, since it natively supports `predict_proba()` —
  required for per-label threshold tuning in the next step

In [10]:
from sklearn.calibration import CalibratedClassifierCV

# Calibrate the base LinearSVC BEFORE wrapping in OneVsRestClassifier —
# CalibratedClassifierCV only supports single-label targets, so it must
# calibrate each binary (one-vs-rest) sub-problem individually, not the
# multi-label problem as a whole.
calibrated_svc = CalibratedClassifierCV(
    LinearSVC(class_weight="balanced", max_iter=2000, C=0.5, random_state=42),
    method="sigmoid",
    cv=3
)

svc_calibrated = OneVsRestClassifier(calibrated_svc)
svc_calibrated.fit(X_train_vec, y_train)

val_probs = svc_calibrated.predict_proba(X_val_vec)
print("Calibrated SVC probabilities generated.")
print(f"Shape: {val_probs.shape}")

Calibrated SVC probabilities generated.
Shape: (31915, 6)


In [11]:
# Choosing threshold that maximize the individual label F1 score on the validation set.
# As the default 0.5 threshold is not optimal, especially the severely imbalanced classes.
# Threshold ranging from 0.2 to 0.8 with 0.05 increment
best_thresholds = {}
print("Optimal Decision Thresholds:")
for i, col in enumerate(label_cols):
    best_th = 0.5
    best_f1 = 0
    for thresh in np.arange(0.2, 0.8, 0.05):
        pred = (val_probs[:, i] >= thresh).astype(int)
        score = f1_score(y_val[:, i], pred, zero_division=0)
        if score > best_f1:
            best_f1 = score
            best_th = thresh

    best_thresholds[col] = round(best_th, 2)
    print(f"  {col:<15}: Threshold = {best_th:.2f} (F1 = {best_f1:.4f})")

Optimal Decision Thresholds:
  toxic          : Threshold = 0.35 (F1 = 0.7828)
  severe_toxic   : Threshold = 0.20 (F1 = 0.4668)
  obscene        : Threshold = 0.25 (F1 = 0.8044)
  threat         : Threshold = 0.20 (F1 = 0.4667)
  insult         : Threshold = 0.20 (F1 = 0.7334)
  identity_hate  : Threshold = 0.20 (F1 = 0.4621)


In [12]:
# Exporting the artifacts
os.makedirs("../models", exist_ok=True)
joblib.dump(vectorizer, "../models/tfidf_vectorizer.joblib")
joblib.dump(svc_calibrated, "../models/toxicity_classifier.joblib")
joblib.dump(best_thresholds, "../models/optimal_thresholds.joblib")
print("\nSuccessful artifacts Exporting'../models/'")


Successful artifacts Exporting'../models/'


## Applying Model to YouTube Comments

In [13]:
# Load your cleaned comments
yt_df = pd.read_csv("../data/processed/comments_cleaned.csv")

# Transform using the fitted vectorizer
X_yt = vectorizer.transform(yt_df["final_text"].fillna(""))

# Predict probabilities
probs = svc_calibrated.predict_proba(X_yt)

# Apply tuned thresholds
for i, col in enumerate(label_cols):
    yt_df[f"score_{col}"] = probs[:, i]
    yt_df[f"is_{col}"] = (probs[:, i] >= best_thresholds[col]).astype(int)

yt_df["is_any_toxic"] = (yt_df[[f"is_{c}" for c in label_cols]].sum(axis=1) > 0).astype(int)

print(f"Scored {len(yt_df)} YouTube comments. Flagged as toxic: {yt_df['is_any_toxic'].sum()}")

Scored 12409 YouTube comments. Flagged as toxic: 323


In [14]:
# --- Training the Hinglish classifier (separate from Jigsaw/English model) ---
df_hinglish = pd.read_csv("../data/raw/hinglish_offensive.csv")  # your uploaded train.csv
df_hinglish["label_binary"] = (df_hinglish["label"] == "offensive").astype(int)

with open("../resources/hinglish_stopwords.txt") as f:
    hinglish_stopwords = list(set(line.strip() for line in f if line.strip()))

X_h = df_hinglish["text"]
y_h = df_hinglish["label_binary"]
X_h_train, X_h_val, y_h_train, y_h_val = train_test_split(
    X_h, y_h, test_size=0.2, random_state=42, stratify=y_h
)

hinglish_vectorizer = TfidfVectorizer(max_features=3000, stop_words=hinglish_stopwords, ngram_range=(1, 2))
X_h_train_vec = hinglish_vectorizer.fit_transform(X_h_train)
X_h_val_vec = hinglish_vectorizer.transform(X_h_val)

model_hinglish = LogisticRegression(class_weight="balanced", max_iter=1000)
model_hinglish.fit(X_h_train_vec, y_h_train)

y_h_pred = model_hinglish.predict(X_h_val_vec)
print(classification_report(y_h_val, y_h_pred))

              precision    recall  f1-score   support

           0       0.92      0.99      0.95       135
           1       0.99      0.91      0.95       139

    accuracy                           0.95       274
   macro avg       0.95      0.95      0.95       274
weighted avg       0.96      0.95      0.95       274



c:\Users\sharm\anaconda3\envs\venv\Lib\site-packages\sklearn\feature_extraction\text.py:412: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['91', 'don', 'mon'] not in stop_words.
  warnings.warn(


#### Observation: Hinglish Classifier
- Trained on a 1.3k dataset, a classifier (offensiv/not-offensive). Combining English-Hinglish stopword.
- Achieved `0.95` Macro F1  on validation. Strong given small dataset.
- The gap between train and validation Macro F1 score is `0.44`  which is a healthy gap. The model generalizes well within the dataset's distribution not memorization.

In [15]:
# Saving Hinglish classifier artifacts
joblib.dump(hinglish_vectorizer, "../models/hinglish_vectorizer.joblib")
joblib.dump(model_hinglish, "../models/hinglish_classifier.joblib")
print("Saved Hinglish classifier and vectorizer.")

Saved Hinglish classifier and vectorizer.


## Comment Routing & Scoring

In [16]:
def score_comment(row):
    text = row["final_text"] if pd.notna(row["final_text"]) else ""

    if row["language_status"] == "likely_misdetected_hinglish":
        vec = hinglish_vectorizer.transform([text])
        proba = model_hinglish.predict_proba(vec)[0][1]
        return pd.Series({
            "model_used": "hinglish",
            "hinglish_offensive_score": proba,
            "is_any_toxic_routed": int(proba >= 0.5)
        })
    else:
        vec = vectorizer.transform([text])
        probs = svc_calibrated.predict_proba(vec)[0]
        is_toxic = int(any(probs[i] >= best_thresholds[col] for i, col in enumerate(label_cols)))
        return pd.Series({
            "model_used": "jigsaw",
            "hinglish_offensive_score": None,
            "is_any_toxic_routed": is_toxic
        })

routed_results = yt_df.apply(score_comment, axis=1)
yt_df["model_used"] = routed_results["model_used"]
yt_df["hinglish_offensive_score"] = routed_results["hinglish_offensive_score"]
yt_df["is_any_toxic_routed"] = routed_results["is_any_toxic_routed"]

print(f"Routed to Hinglish model: {(yt_df['model_used']=='hinglish').sum()}")
print(f"Routed to Jigsaw model:   {(yt_df['model_used']=='jigsaw').sum()}")
print(f"\nFlagged toxic (unrouted): {yt_df['is_any_toxic'].sum()}")

print(f"Flagged toxic (routed):   {yt_df['is_any_toxic_routed'].sum()}")

Routed to Hinglish model: 5576
Routed to Jigsaw model:   6833

Flagged toxic (unrouted): 323
Flagged toxic (routed):   742


#### Observation: Routed vs. Unrouted Scoring
- Without language-aware routing, 323 of 12,521 comments (2.6%) were flagged as toxic using only the English-trained Jigsaw/SVC model on every comment regardless of language.
- With routing — Hinglish-flagged comments scored by a dedicated Hinglish classifier genuinely non-English comments translated before scoring, and everything else scored by the Jigsaw/SVC model — 742 comments (5.9%) were flagged as toxic, a 130% increase, revealing 419 toxic comments the English-only pipeline was blind to.
- 5,576 comments (44.5% of the dataset) were routed to the Hinglish model; 6,833 (54.6%) were scored by the Jigsaw/SVC model (including the smaller `likely_genuine_non_english` bucket, translated first).

## Conclusion
This confirms that applying an English-trained toxicity classifier uniformly to an Indian YouTube channel's comments substantially under counts real toxicity, because large share of comments being Hinglish. This is resolve by a lightweight classifier dedicated to hinglish.

In [18]:
yt_df.to_csv("../data/processed/comments_scored.csv", index=False)
print(f"Saved {len(yt_df)} scored comments.")

Saved 12409 scored comments.
